#### Getting started with Langchain

In [1]:
import langchain

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
os.environ["Google_API_KEY"] = os.getenv("Google_API_KEY")

### Example 1 : Simple LLM with Streaming

In [4]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage


###  1 way for the model development

In [5]:
model=init_chat_model("groq:llama-3.3-70b-versatile")

### 2 way for the model development

In [6]:
from langchain_groq import ChatGroq
llm=ChatGroq(model="llama-3.3-70b-versatile")


#### Message Creation

In [7]:
message=[
    SystemMessage(content="You are a helpful assistant that translates English to French."),
    HumanMessage(content="Translate this sentence from English to French. I love programming.")
    
]

#### Invoke the model

In [8]:
response=model.invoke(message)
print(response)

content='The translation of "I love programming" from English to French is:\n\nJ\'adore la programmation.\n\nHere\'s a breakdown:\n- "I" is translated to "Je", but in informal speech, it\'s often reduced to "J\'" before a vowel.\n- "love" is translated to "aime" or "adore" depending on the intensity of the feeling. In this case, "adore" is used for a stronger affection.\n- "programming" is translated to "la programmation".' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 58, 'total_tokens': 164, 'completion_time': 0.358651409, 'completion_tokens_details': None, 'prompt_time': 0.002930777, 'prompt_tokens_details': None, 'queue_time': 0.053866803, 'total_time': 0.361582186}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019fc1c7-f5ce-7243-a5f5-d66ec3e5c847-0' tool_calls=[] invalid_tool_calls=[] u

### Streaming Example

In [9]:
for chunk in model.stream(message):
    print(chunk.content, end="",flush=True)

The translation of the sentence "I love programming" from English to French is:

J'adore la programmation.

Here's a breakdown:
- "I" is translated to "Je", but in this case, it's "J'" because it's followed by a vowel.
- "love" is translated to "adore".
- "programming" is translated to "la programmation".

### Dynamic Prompts

In [10]:
from langchain_core.prompts import ChatPromptTemplate

## creating a prompt template
translation_prompt=ChatPromptTemplate.from_messages([
    ("system","You are a helpful assistant that translates English to French.translate the following {sentence} to {source_language} to {target_language}.Maintain the tone and style of the original text."),
    ("human","{text}")  ])

## Using the template to create a prompt
prompt=translation_prompt.format_prompt(sentence="Translate this sentence from English to French. I love programming.",
                                        source_language="English",
                                        target_language="French",
                                        text="I love programming.")
print (prompt)

messages=[SystemMessage(content='You are a helpful assistant that translates English to French.translate the following Translate this sentence from English to French. I love programming. to English to French.Maintain the tone and style of the original text.', additional_kwargs={}, response_metadata={}), HumanMessage(content='I love programming.', additional_kwargs={}, response_metadata={})]


In [11]:
translated_response=model.invoke(prompt)
print(translated_response)

content="J'adore la programmation." additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 80, 'total_tokens': 89, 'completion_time': 0.038182494, 'completion_tokens_details': None, 'prompt_time': 0.004174367, 'prompt_tokens_details': None, 'queue_time': 0.056213812, 'total_time': 0.042356861}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019fc1c7-f9af-76f1-a847-6c9fb581a397-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 80, 'output_tokens': 9, 'total_tokens': 89}


## Building your first Chain

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

def create_translation_chain():

    # Prompt for story generation
    story_prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "You are a creative story writer. Write a story in {target_language} based on the following outline: {text}"
        ),
        (
            "human",
            "Theme: {theme}\nMain Character: {main_character}\nPlot: {plot}"
        )
    ])

    # template for story generation
    analysis_prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "You are a literary analyst. Analyze the following story in {target_language} and provide insights on its themes, characters, and plot."
        ),
        (
            "human",
            "{story}"
        )
    ])
    story_chain=(
        story_prompt | model | StrOutputParser() 
    )

    #Create a function to pass the story to the analysis chain
    def analysis_story(story_text):
        return {
            "story": story_text,
            "target_language": "Marathi"  # You can change this to any target language you want
        }

    
    analysis_chain=(
     story_chain|RunnableLambda(analysis_story) |analysis_prompt | model | StrOutputParser()
    )
    return analysis_chain

In [19]:
chain=create_translation_chain()
chain

ChatPromptTemplate(input_variables=['main_character', 'plot', 'target_language', 'text', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['target_language', 'text'], input_types={}, partial_variables={}, template='You are a creative story writer. Write a story in {target_language} based on the following outline: {text}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['main_character', 'plot', 'theme'], input_types={}, partial_variables={}, template='Theme: {theme}\nMain Character: {main_character}\nPlot: {plot}'), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False,

In [20]:
result = chain.invoke({
    "theme": "A young girl discovers a hidden world of magic",
    "main_character": "Lila, a curious and adventurous 12-year-old girl",
    "plot": "She stumbles upon a secret portal in her grandmother's attic and finds herself transported to a realm where magic is real.",
    "target_language": "marathi",   
    "text": "Write a story in marathi based on the above outline."
})  
result

'ही कथा एका १२ वर्षांच्या मुलगी, लिला, आणि तिच्या जादुई प्रवासाची आहे. लिला ही एक उत्साही आणि साहसी मुलगी आहे जी नेहमी नवीन गोष्टींबद्दल शोधत असते. ती तिच्या आजोबांच्या घरी पुराणी वस्तू शोधत असताना तिला एक गुप्त दаровात सापडते, ज्यामुळे ती एका नवीन जगात जाते.\n\nही कथा अनेक थीम्सवर भर देते, जसे की:\n\n१. **साहस आणि उत्साह**: लिला ही एक साहसी मुलगी आहे जी नेहमी नवीन गोष्टींबद्दल शोधत असते. तिच्या साहसामुळे तिला एका नवीन जगात जाण्याची संधी मिळते.\n२. **जादू आणि कल्पनाशक्ती**: कथा जादू आणि कल्पनाशक्तीच्या महत्त्वावर भर देते. लिला जादू शिकून घेते आणि तिच्या जादूच्या मदतीने लोकांची मदत करते.\n३. **कुटुंब आणि परंपरा**: कथा कुटुंब आणि परंपरेच्या महत्त्वावर भर देते. लिला तिच्या आजोबांच्या घरी पुराणी वस्तू शोधत असताना तिला एक गुप्त दаровात सापडते, ज्यामुळे तिला तिच्या कुटुंबाच्या इतिहासाची माहिती मिळते.\n४. **नैतिकता आणि जबाबदारी**: कथा नैतिकता आणि जबाबदारीच्या महत्त्वावर भर देते. लिलाला तिच्या आजीने सांगितले की तिला जादू वाईट कामासाठी वापरू नये.\n\nपात्रे:\n\n१. **लिला**: लिला ही एक १२ वर्षांच

'ही कथा एका १२ वर्षांच्या मुलगी, लिला, आणि तिच्या जादुई प्रवासाची आहे. लिला ही एक उत्साही आणि साहसी मुलगी आहे जी नेहमी नवीन गोष्टींबद्दल शोधत असते. ती तिच्या आजोबांच्या घरी पुराणी वस्तू शोधत असताना तिला एक गुप्त दаровात सापडते, ज्यामुळे ती एका नवीन जगात जाते.\n\nही कथा अनेक थीम्सवर भर देते, जसे की:\n\n१. **साहस आणि उत्साह**: लिला ही एक साहसी मुलगी आहे जी नेहमी नवीन गोष्टींबद्दल शोधत असते. तिच्या साहसामुळे तिला एका नवीन जगात जाण्याची संधी मिळते.\n२. **जादू आणि कल्पनाशक्ती**: कथा जादू आणि कल्पनाशक्तीच्या महत्त्वावर भर देते. लिला जादू शिकून घेते आणि तिच्या जादूच्या मदतीने लोकांची मदत करते.\n३. **कुटुंब आणि परंपरा**: कथा कुटुंब आणि परंपरेच्या महत्त्वावर भर देते. लिला तिच्या आजोबांच्या घरी पुराणी वस्तू शोधत असताना तिला एक गुप्त दаровात सापडते, ज्यामुळे तिला तिच्या कुटुंबाच्या इतिहासाची माहिती मिळते.\n४. **नैतिकता आणि जबाबदारी**: कथा नैतिकता आणि जबाबदारीच्या महत्त्वावर भर देते. लिलाला तिच्या आजीने सांगितले की तिला जादू वाईट कामासाठी वापरू नये.\n\nपात्रे:\n\n१. **लिला**: लिला ही एक १२ वर्षांची उत्साही आणि साहसी मुलगी आहे. ती नेहमी नवीन गोष्टींबद्दल शोधत असते आणि तिच्या जादूच्या मदतीने लोकांची मदत करते.\n२. **आजी**: आजी ही लिलाच्या आजीची भूमिका बजावते. ती लिलाला जादू शिकवते आणि तिला नैतिकता आणि जबाबदारीच्या महत्त्वावर भर देते.\n३. **आजोबा**: आजोबा ही लिलाच्या आजोबांची भूमिका बजावते. त्यांनी एक गुप्त दаровात तयार केले होते ज्यामुळे लिलाला एका नवीन जगात जाण्याची संधी मिळते.\n\nकथेचा मुख्य संदेश म्हणजे जादू आणि कल्पनाशक्ती ही एक शक्तिशाली साधने आहेत ज्यांचा वापर चांगल्या कामासाठी केला पाहिजे. कथा नैतिकता आणि जबाबदारीच्या महत्त्वावर भर देते आणि वाचकांना जादू आणि कल्पनाशक्तीच्या साहाय्याने लोकांची मदत करण्याचे प्रोत्साहन देते.'